# smartreact — Examples

This notebook demonstrates the core functionality of the `smartreact` package:

1. Key classification of molecules
2. Single-pair reaction enumeration
3. Batch enumeration with multiple pairs
4. Filtering by specific reaction types
5. Visualising reactants and products

In [1]:
from smartreact import KeyGenerator, ReactionEnumerator
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

## 1. Key Classification

The `KeyGenerator` classifies molecules by their reactive functional groups.
Each SMILES is matched against a set of SMARTS rules to produce category,
subcategory, and subsubcategory labels.

In [2]:
keygen = KeyGenerator()

molecules = {
    "bromobenzene":      "c1ccc(Br)cc1",
    "phenylboronic acid": "c1ccc(B(O)O)cc1",
    "ethylamine":        "CCN",
    "acetic acid":       "CC(=O)O",
    "iodobenzene":       "c1ccc(I)cc1",
}

for name, smi in molecules.items():
    result = keygen.classify(smi)
    cats, subs, subsubs = result.categories()
    print(f"{name} ({smi})")
    print(f"  Categories:       {sorted(cats)}")
    print(f"  Subcategories:    {sorted(subs)}")
    print(f"  Subsubcategories: {sorted(subsubs)}")
    print()

bromobenzene (c1ccc(Br)cc1)
  Categories:       ['X']
  Subcategories:    ['X_Aromatic']
  Subsubcategories: ['X-Bromide_Phe']

phenylboronic acid (c1ccc(B(O)O)cc1)
  Categories:       ['Boronic']
  Subcategories:    ['Boronic_Aromatic']
  Subsubcategories: ['Boronic_Aromatic']

ethylamine (CCN)
  Categories:       ['Amine']
  Subcategories:    ['Amine_Aliphatic']
  Subsubcategories: ['Amine_Primary_SaturatedAliphatic']

acetic acid (CC(=O)O)
  Categories:       ['Acid']
  Subcategories:    ['Acid_Aliphatic']
  Subsubcategories: ['Acid_SaturatedAliphatic']

iodobenzene (c1ccc(I)cc1)
  Categories:       ['X']
  Subcategories:    ['X_Aromatic']
  Subsubcategories: ['X-Iodide_Phe']



## 2. Single-Pair Reaction Enumeration

The `ReactionEnumerator` uses SMARTS reaction templates paired with Key
filtering to enumerate plausible products from two reactants.

In [3]:
enumerator = ReactionEnumerator()

# Suzuki coupling: bromobenzene + phenylboronic acid -> biphenyl
results = enumerator.enumerate_pair("c1ccc(Br)cc1", "c1ccc(B(O)O)cc1")

for r in results:
    print(r)
    print()

Reaction: suzuki
Reactant A: c1ccc(B(O)O)cc1
Reactant B: c1ccc(Br)cc1
Products (1):
  1. c1ccc(-c2ccccc2)cc1



### Convenience helper: just the products

In [4]:
products = enumerator.products_for_pair("c1ccc(Br)cc1", "c1ccc(B(O)O)cc1")
print("Unique products:", products)

Unique products: ['c1ccc(-c2ccccc2)cc1']


## 3. Batch Enumeration

Process many reactant pairs at once. Use `parallel=True` and set `n_cores`
on the enumerator constructor for large batches.

For very large datasets, use `enumerate_pairs_lazy` to stream results without
materialising everything in memory at once.

In [5]:
pairs = [
    ("c1ccc(Br)cc1", "c1ccc(B(O)O)cc1"),  # Suzuki
    ("c1ccc(I)cc1", "CCN"),                  # SNAr / Buchwald
    ("CC(=O)O", "CCNC"),                     # Amide coupling
]

# n_cores is set on the constructor, not on enumerate_pairs
enumerator_parallel = ReactionEnumerator(n_cores=4)
all_results = enumerator_parallel.enumerate_pairs(pairs, parallel=True)

print(f"Total reaction results: {len(all_results)}")
for r in all_results:
    print(f"  {r.reaction_name}: {r.reactant_a} + {r.reactant_b} -> {r.products}")

Total reaction results: 3
  suzuki: c1ccc(B(O)O)cc1 + c1ccc(Br)cc1 -> ['c1ccc(-c2ccccc2)cc1']
  B-H: CCN + c1ccc(I)cc1 -> ['CCNc1ccccc1']
  amide_coupling: CC(=O)O + CCNC -> ['CCN(C)C(C)=O']


### Streaming large datasets with `enumerate_pairs_lazy`

`enumerate_pairs_lazy` yields results incrementally so peak memory is bounded
by `chunk_size` (default 50 000 pairs) rather than the total input size.
The `pairs` argument can be any iterable, including a generator.

In [6]:
# enumerate_pairs_lazy yields ReactionResult objects one chunk at a time
collected = []
for result in enumerator_parallel.enumerate_pairs_lazy(pairs, parallel=True, chunk_size=1_000):
    collected.append(result)

print(f"Total reaction results: {len(collected)}")
for r in collected:
    print(f"  {r.reaction_name}: {r.reactant_a} + {r.reactant_b} -> {r.products}")

# For truly large inputs, avoid materialising at all:
# for result in enumerator_parallel.enumerate_pairs_lazy(huge_generator, parallel=True):
#     write_to_disk(result)

Total reaction results: 3
  suzuki: c1ccc(B(O)O)cc1 + c1ccc(Br)cc1 -> ['c1ccc(-c2ccccc2)cc1']
  B-H: CCN + c1ccc(I)cc1 -> ['CCNc1ccccc1']
  amide_coupling: CC(=O)O + CCNC -> ['CCN(C)C(C)=O']


## 4. Filtering by Reaction Type

You can restrict enumeration to specific reaction types.

In [7]:
# List every reaction template currently available
available = sorted({t.name for t in ReactionEnumerator().templates})
print(f"{len(available)} available reactions:")
for name in available:
    print(f"  - {name}")

36 available reactions:
  - ART
  - B-H
  - C_C_decarboxylation
  - amide_coupling
  - amine_acetylation
  - amine_sulfonation
  - chan_lam
  - cross_electrophile_coupling
  - deoxygenative_coupling
  - ester_schotten_baumann
  - ester_sulfonic_schotten_baumann
  - esterification
  - heck
  - horner_wadsworth_emmons
  - imidazole_Xketone_synthesis
  - imidazole_condensation_acid
  - imidazole_condensation_amine
  - mitsunobu
  - negishi
  - oxadiazole_condensation
  - pinacolatoborylation
  - reductive_amination_aldehyde
  - reductive_amination_ketone
  - sn2_nheterocycle
  - snar_alcohol
  - snar_amine
  - snar_thiol
  - sonogashira
  - suzuki
  - tetrazole_synthesis
  - triazole_synthesis_1
  - triazole_synthesis_2
  - ullmann_X
  - ullmann_phenol
  - urea_formation
  - williamson


In [8]:
# Only Suzuki coupling
suzuki_only = ReactionEnumerator(reaction_list=["suzuki"])
results = suzuki_only.enumerate_pair("c1ccc(Br)cc1", "c1ccc(B(O)O)cc1")
print(f"Suzuki-only results: {len(results)}")
for r in results:
    print(f"  {r.products}")

Suzuki-only results: 1
  ['c1ccc(-c2ccccc2)cc1']


## 5. Visualising Reactants and Products

This is a simple convenience snippet for quickly visualizing reactants and products using RDKit drawing utilities.
You can copy and adapt this code for your own use; for more advanced visualization, consider using your preferred code/tools.

In [ ]:
results = enumerator.enumerate_pair("c1ccc(Br)cc1", "c1ccc(B(O)O)cc1")

for r in results:
    mols = [Chem.MolFromSmiles(s) for s in [r.reactant_a, r.reactant_b] + r.products if s]
    mols = [m for m in mols if m is not None]
    legends = (
        [f"Reactant A", f"Reactant B"]
        + [f"Product {i+1}" for i in range(len(r.products))]
    )
    print(f"Reaction: {r.reaction_name}")
    img = Draw.MolsToGridImage(mols, legends=legends[:len(mols)], molsPerRow=4, subImgSize=(300, 250))
    display(img)